# --------------------------------------
# 1. Install & Environment Check
# --------------------------------------

In [ ]:
import os
import json
import datetime
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("⚠️ WARNING: Training will be much slower without a GPU.")

PyTorch: 2.8.0+cu126
CUDA available: True


# --------------------------------------
# 2. (Optional) Mount Google Drive for persistence
# --------------------------------------
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# Change this to a Drive path if you want persistence:
DATASETS_BASE = "/content/datasets"
os.makedirs(DATASETS_BASE, exist_ok=True)

# --------------------------------------
# 3. Upload Dataset ZIP
# --------------------------------------

In [ ]:
from google.colab import files
uploaded = files.upload()

zip_name = next(iter(uploaded.keys()))
print("Uploaded ZIP:", zip_name)

Saving dataset.zip to dataset (2).zip
Uploaded ZIP: dataset (2).zip


# --------------------------------------
# 4. Automatic Dataset Versioning
# --------------------------------------

In [ ]:
zip_stem = os.path.splitext(zip_name)[0]  # e.g. "robot_layouts"
print("Dataset base name:", zip_stem)

def get_next_version_dir(base_dir: str, stem: str) -> str:
    """
    Scan base_dir for existing <stem>_vXXX folders and return a new folder path
    with the next version number.
    """
    existing = []
    prefix = f"{stem}_v"
    for name in os.listdir(base_dir):
        if name.startswith(prefix):
            try:
                suffix = name[len(prefix):]  # "001", "002", etc.
                existing.append(int(suffix))
            except ValueError:
                continue

    next_version = (max(existing) + 1) if existing else 1
    version_str = f"{next_version:03d}"
    version_folder_name = f"{stem}_v{version_str}"
    return os.path.join(base_dir, version_folder_name), next_version

DATASET_ROOT, version_number = get_next_version_dir(DATASETS_BASE, zip_stem)
os.makedirs(DATASET_ROOT, exist_ok=True)

print(f"✔ Creating dataset version: v{version_number:03d}")
print("Version folder:", DATASET_ROOT)

Dataset base name: dataset (2)
✔ Creating dataset version: v001
Version folder: /content/datasets/dataset (2)_v001


# --------------------------------------
# 5. Extract ZIP into Versioned Folder
# --------------------------------------

In [ ]:
import zipfile

with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(DATASET_ROOT)

print("Extracted into:", DATASET_ROOT)
print("Top-level contents:", os.listdir(DATASET_ROOT))

Extracted into: /content/datasets/dataset (2)_v001
Top-level contents: ['README.dataset.txt', 'README.roboflow.txt', 'test', 'data.yaml', 'valid', 'train']


# --------------------------------------
# 6. Locate data.yaml inside this version
# --------------------------------------

In [ ]:
data_yaml_path = None
for root, dirs, files in os.walk(DATASET_ROOT):
    for f in files:
        if f == "data.yaml":
            data_yaml_path = os.path.join(root, f)

if data_yaml_path is None:
    raise FileNotFoundError("❌ data.yaml not found in extracted dataset version!")
else:
    print("✔ Using data.yaml:", data_yaml_path)

✔ Using data.yaml: /content/datasets/dataset (2)_v001/data.yaml


# --------------------------------------
# 7. Write metadata.json for this dataset version
# --------------------------------------

In [ ]:
metadata = {
    "zip_name": zip_name,
    "dataset_base_name": zip_stem,
    "version_number": version_number,
    "dataset_root": DATASET_ROOT,
    "data_yaml_path": data_yaml_path,
    "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",
}

metadata_path = os.path.join(DATASET_ROOT, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print("✔ Wrote metadata:", metadata_path)

# Optional: also record / update a "latest" pointer file
latest_info_path = os.path.join(DATASETS_BASE, f"{zip_stem}_latest.json")
with open(latest_info_path, "w") as f:
    json.dump(metadata, f, indent=2)
print("✔ Updated latest dataset pointer:", latest_info_path)

✔ Wrote metadata: /content/datasets/dataset (2)_v001/metadata.json
✔ Updated latest dataset pointer: /content/datasets/dataset (2)_latest.json


/tmp/ipython-input-2821888567.py:7: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at_utc": datetime.datetime.utcnow().isoformat() + "Z",


# --------------------------------------
# 8. Load Base Model
# --------------------------------------

In [ ]:
model = YOLO("yolov8s.pt")

# --------------------------------------
# 9. Hyperparameter Tuning (Optional)
# --------------------------------------

In [ ]:
# You can skip or reduce epochs to save time
model.tune(
    data=data_yaml_path,
    epochs=20,
    imgsz=512,
    name=f"{zip_stem}_tuner_v{version_number:03d}",
    save=True,
)

Tuner: Initialized Tuner instance with 'tune_dir=/content/runs/detect/dataset (2)_tuner_v001'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/10 with hyperparameters: {'lr0': 0.01, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'box': 7.5, 'cls': 0.5, 'dfl': 1.5, 'hsv_h': 0.015, 'hsv_s': 0.7, 'hsv_v': 0.4, 'degrees': 0.0, 'translate': 0.1, 'scale': 0.5, 'shear': 0.0, 'perspective': 0.0, 'flipud': 0.0, 'fliplr': 0.5, 'bgr': 0.0, 'mosaic': 1.0, 'mixup': 0.0, 'cutmix': 0.0, 'copy_paste': 0.0, 'close_mosaic': 10}
Saved /content/runs/detect/dataset (2)_tuner_v001/tune_scatter_plots.png
Saved /content/runs/detect/dataset (2)_tuner_v001/tune_fitness.png

Tuner: 1/10 iterations complete ✅ (138.40s)
Tuner: Results saved to /content/runs/detect/dataset (2)_tuner_v001
Tuner: Best fitness=0.67956 observed at iteration 1
Tuner: Best fitness metrics are {'metrics/precision(B)':

# --------------------------------------
# 10. Load Best Hyperparameters from Tuner
# --------------------------------------

In [ ]:
tuner_run_dir = f"/content/runs/detect/{zip_stem}_tuner_v{version_number:03d}"
best_cfg = os.path.join(tuner_run_dir, "best_hyperparameters.yaml")

if not os.path.exists(best_cfg):
    raise FileNotFoundError(f"❌ best_hyperparameters.yaml not found at {best_cfg}")

print("✔ Loaded tuner config:", best_cfg)

✔ Loaded tuner config: /content/runs/detect/dataset (2)_tuner_v001/best_hyperparameters.yaml


# --------------------------------------
# 11. Final Training Using Tuned Hyperparameters
# --------------------------------------

In [ ]:
import yaml

# --- Fix invalid hyperparameter types ---
with open(best_cfg, "r") as f:
    cfg_dict = yaml.safe_load(f)

if "close_mosaic" in cfg_dict:
    val = cfg_dict["close_mosaic"]
    if isinstance(val, float):
        print(f"⚠️ Fixing close_mosaic: converting {val} -> {int(val)}")
        cfg_dict["close_mosaic"] = int(val)

# write back the fixed YAML
with open(best_cfg, "w") as f:
    yaml.dump(cfg_dict, f)

print("✔ Fixed hyperparameters written back to:", best_cfg)

⚠️ Fixing close_mosaic: converting 10.0 -> 10
✔ Fixed hyperparameters written back to: /content/runs/detect/dataset (2)_tuner_v001/best_hyperparameters.yaml


In [ ]:
train_run_name = f"{zip_stem}_yolov8s_v{version_number:03d}"

results = model.train(
    data=data_yaml_path,
    cfg=best_cfg,
    epochs=120,
    imgsz=512,
    batch=16,
    name=train_run_name,
    plots=True,
)

print("✔ Training complete. Output saved to:")
print(f"  /content/runs/detect/{train_run_name}")

# --------------------------------------
# 12. Count-based evaluation on validation set
# --------------------------------------

In [ ]:
import os, glob
from IPython.display import Image, display

run_dir = f"/content/runs/detect/{train_run_name}"
print("Using run directory:", run_dir)

key_plots = [
    "results.png",            # losses + mAP curves
    "confusion_matrix.png",   # confusion matrix
    "PR_curve.png",           # precision-recall curve
    "F1_curve.png",           # F1 vs confidence
    "P_curve.png",            # precision vs confidence
    "R_curve.png",            # recall vs confidence
]

for fname in key_plots:
    fpath = os.path.join(run_dir, fname)
    if os.path.exists(fpath):
        print("Showing:", fname)
        display(Image(filename=fpath))
    else:
        print("Not found:", fname)

# --------------------------------------
# 13. Count-based evaluation on validation set
# --------------------------------------


In [ ]:
import glob
import os
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

# Make sure best_weights_path and DATASET_ROOT already exist
best_weights_path = f"/content/runs/detect/{train_run_name}/weights/best.pt"
print("Using weights:", best_weights_path)

model = YOLO(best_weights_path)

# Find validation images for this dataset version
val_imgs = glob.glob(os.path.join(DATASET_ROOT, "**/valid/images/*.*"), recursive=True)
print("Found", len(val_imgs), "validation images")

def label_path_from_image(img_path: str) -> str:
    """
    Convert YOLO image path to corresponding label path.
    Example: .../images/img01.jpg -> .../labels/img01.txt
    """
    label_path = img_path.replace(os.sep + "images" + os.sep, os.sep + "labels" + os.sep)
    base, _ = os.path.splitext(label_path)
    return base + ".txt"

gt_counts = []
pred_counts = []
image_ids = []

conf_threshold = 0.25

# Use stream=True to avoid storing all predictions in memory at once
pred_stream = model.predict(
    source=val_imgs,
    imgsz=512,
    conf=conf_threshold,
    stream=True,
    verbose=False,
)

for img_path, pred in zip(val_imgs, pred_stream):
    # --- Ground truth count ---
    lbl_path = label_path_from_image(img_path)
    if os.path.exists(lbl_path):
        with open(lbl_path, "r") as f:
            lines = [ln.strip() for ln in f.readlines() if ln.strip()]
        gt_count = len(lines)
    else:
        gt_count = 0  # no label file means zero GT objects

    # --- Predicted count ---
    # pred.boxes is a Boxes object; len(pred.boxes) is number of detections
    pred_count = len(pred.boxes)

    gt_counts.append(gt_count)
    pred_counts.append(pred_count)
    image_ids.append(os.path.basename(img_path))

gt_counts = np.array(gt_counts)
pred_counts = np.array(pred_counts)
errors = pred_counts - gt_counts

# Basic counting metrics
mae = np.mean(np.abs(errors))
rmse = np.sqrt(np.mean(errors ** 2))
exact_match = np.mean(errors == 0) * 100.0

print(f"MAE (|pred - gt|): {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"Exact count match: {exact_match:.2f}% of images")

# --------------------------------------
#14. Parity plot: ground-truth count vs predicted count
# --------------------------------------




In [ ]:
plt.figure()
plt.scatter(gt_counts, pred_counts)
max_count = max(gt_counts.max(), pred_counts.max(), 1)
plt.plot([0, max_count], [0, max_count], linestyle="--")
plt.xlabel("Ground Truth Count")
plt.ylabel("Predicted Count")
plt.title("Robot Count Parity Plot (Validation Set)")
plt.grid(True)
plt.show()

# --------------------------------------
#15. Error distribution: (predicted - ground_truth)
# --------------------------------------

In [ ]:
plt.figure()
plt.hist(errors, bins=range(int(errors.min()) - 1, int(errors.max()) + 2))
plt.xlabel("Prediction Error (predicted - ground_truth)")
plt.ylabel("Number of Images")
plt.title("Robot Count Error Distribution (Validation Set)")
plt.grid(True)
plt.show()

# --------------------------------------
# 16. Inference on Sample Validation Images
# --------------------------------------


In [ ]:
import glob
import os
import random
from ultralytics import YOLO
from IPython.display import Image, display

best_weights_path = f"/content/runs/detect/{train_run_name}/weights/best.pt"

print("Using best weights from:", best_weights_path)
model = YOLO(best_weights_path)

Using best weights from: /content/runs/detect/dataset (2)_yolov8s_v001/weights/best.pt


In [ ]:
val_images = glob.glob(os.path.join(DATASET_ROOT, "**/valid/images/*.*"), recursive=True)
print("Found", len(val_images), "validation images.")

if len(val_images) == 0:
    raise RuntimeError("❌ No validation images found! Check dataset structure.")

# Random sample (default = 5)
sample_size = min(5, len(val_images))
sample_images = random.sample(val_images, sample_size)
print("Running inference on:", sample_images)


Found 9 validation images.
Running inference on: ['/content/datasets/dataset (2)_v001/valid/images/layout_001_tile_r001_c002_png.rf.63b4c96cd969640d2934089bf0905a83.jpg', '/content/datasets/dataset (2)_v001/valid/images/layout_008_tile_r001_c000_png.rf.b327a5ce3cb69bfe5501834b6345e5b8.jpg', '/content/datasets/dataset (2)_v001/valid/images/layout_003_tile_r000_c000_png.rf.42e9a5044c193e9b6dafc85f3089ce39.jpg', '/content/datasets/dataset (2)_v001/valid/images/layout_001_tile_r001_c001_png.rf.c4c194085ceac1aa1b5f8419b87a613c.jpg', '/content/datasets/dataset (2)_v001/valid/images/layout_002_tile_r000_c002_png.rf.79e831606cb1265d9db258b5113d2383.jpg']


In [ ]:
output_dir = f"runs/vis/{zip_stem}_samples_v{version_number:03d}"

results = model.predict(
    source=sample_images,
    imgsz=512,
    conf=0.25,               # adjustable
    save=True,
    project="runs/vis",
    name=f"{zip_stem}_samples_v{version_number:03d}",
    exist_ok=True,
)

print("✔ Inference complete.")
print("Images saved to:", output_dir)


0: 512x512 11 robots, 7.9ms
1: 512x512 8 robots, 7.9ms
2: 512x512 8 robots, 7.9ms
3: 512x512 11 robots, 7.9ms
4: 512x512 8 robots, 7.9ms
Speed: 1.3ms preprocess, 7.9ms inference, 1.0ms postprocess per image at shape (1, 3, 512, 512)
Results saved to /content/runs/vis/dataset (2)_samples_v001
✔ Inference complete.
Images saved to: runs/vis/dataset (2)_samples_v001


In [ ]:
result_images = glob.glob(os.path.join(output_dir, "*.*g"))

print("Showing annotated detections:")
for img_path in result_images:
    display(Image(filename=img_path))

# ==================================================
# Export YOLOv8 model to ONNX for application use
# ==================================================

In [ ]:
!pip install -q onnx onnxruntime onnxslim

from ultralytics import YOLO
import os, glob, re
from google.colab import files

best_weights_path = f"/content/runs/detect/{train_run_name}/weights/best.pt"
print("Using best weights:", best_weights_path)

if not os.path.exists(best_weights_path):
    raise FileNotFoundError(f"❌ best.pt not found at {best_weights_path}")

model = YOLO(best_weights_path)

# --------------------------------------------
# 3. Build a safe export name (no spaces, brackets)
# --------------------------------------------
def make_safe_name(name: str) -> str:
    # Keep letters, numbers, underscore, dash; replace others with underscore
    return re.sub(r"[^A-Za-z0-9_-]", "_", name)

safe_stem = make_safe_name(zip_stem)
export_run_name = f"{safe_stem}_v{version_number:03d}"
export_dir = f"/content/runs/export/{export_run_name}"

print("Export run name:", export_run_name)
print("Export dir will be:", export_dir)

# --------------------------------------------
# 4. Export to ONNX (clean argument set)
# --------------------------------------------
print("\n🔁 Exporting model to ONNX...")

export_result = model.export(
    format="onnx",
    opset=12,            # good compatibility default
    dynamic=True,        # allow variable input sizes
    simplify=True,       # simplify ONNX graph with onnxslim
    imgsz=512,           # match your training imgsz
    project="runs/export",
    name=export_run_name,
)

# ----------------------------------------------
# Download ONNX to your local machine
# ----------------------------------------------

In [ ]:
onnx_files = glob.glob(os.path.join(export_dir, "*.onnx"))
print("\nDiscovered ONNX files:")
for f in onnx_files:
    print("  -", f)

if not onnx_files:
    raise FileNotFoundError("❌ No ONNX files found after export. Check the printed logs above for errors.")

onnx_path = onnx_files[0]
print("\n✔ Will download:", onnx_path)

In [ ]:
print("Starting download...")
onnx_path = "/content/runs/detect/dataset (2)_yolov8s_v001/weights/best.onnx"
files.download(onnx_path)